# 04 — Virtual Memory, Storage, TCP, and Browser Boundaries

**Modules:** `05.01`, `05.02`, `05.03`, `05.04`, `06.01`, `06.04`  
**Prerequisite:** notebook 03; the assembled machine image becomes a file, a mapped page, packet payload, and browser-visible response.  
**Consumes:** `fromthetransistor/03-machine-image.json`  
**Emits:** `fromthetransistor/04-system-image.json`

This integration notebook crosses several boundaries without pretending to implement a production OS, FAT, TCP, or HTML engine. Each miniature interface has explicit width, range, permission, checksum, and length checks. The point is to trace one byte sequence across layers and keep faults attributable to the layer that rejected it.

In [ ]:
from pathlib import Path
import json
import os

ARTIFACT_ROOT = Path(os.environ['COURSE_NOTEBOOK_ARTIFACTS'])
machine = json.loads((ARTIFACT_ROOT / 'fromthetransistor/03-machine-image.json').read_text(encoding='utf-8'))
machine_image = bytes.fromhex(machine['image_hex'])
assert machine['schema_version'] == 1
assert machine['byteorder'] == 'little'
assert len(machine_image) == 14


## Worked integration

A virtual address splits into virtual-page number and page offset; permissions are checked before producing a physical address. A block device rejects partial or out-of-range access; the tiny file layer records a name, block, and exact byte length. The Internet checksum detects this controlled corruption case but is not cryptographic integrity. Finally, the browser boundary refuses a response whose declared body length disagrees with its bytes.

In [ ]:
PAGE_SIZE = 256
page_table = {0x40: {'pfn': 0x02, 'writable': False, 'user': True}}

def translate(virtual_address, access='read', user=True):
    if not 0 <= virtual_address <= 0xFFFF:
        raise MemoryError('virtual address width fault')
    vpn, offset = divmod(virtual_address, PAGE_SIZE)
    entry = page_table.get(vpn)
    if entry is None:
        raise MemoryError('page not present')
    if access == 'write' and not entry['writable']:
        raise PermissionError('write-protected page')
    if user and not entry['user']:
        raise PermissionError('supervisor page')
    return entry['pfn'] * PAGE_SIZE + offset

class BlockDevice:
    def __init__(self, block_count=4, block_size=64):
        self.block_size = block_size
        self.blocks = [bytes(block_size) for _ in range(block_count)]
    def write(self, lba, data):
        if not 0 <= lba < len(self.blocks):
            raise IndexError('LBA out of range')
        if len(data) != self.block_size:
            raise ValueError('writes must be exactly one block')
        self.blocks[lba] = bytes(data)
    def read(self, lba):
        if not 0 <= lba < len(self.blocks):
            raise IndexError('LBA out of range')
        return self.blocks[lba]

class TinyFS:
    def __init__(self, device):
        self.device = device
        self.directory = {}
    def put(self, name, data, lba):
        if not name or '/' in name or len(data) > self.device.block_size:
            raise ValueError('invalid tiny file')
        block = data + bytes(self.device.block_size - len(data))
        self.device.write(lba, block)
        self.directory[name] = {'lba': lba, 'length': len(data)}
    def get(self, name):
        entry = self.directory[name]
        return self.device.read(entry['lba'])[:entry['length']]

def internet_checksum(data):
    if len(data) % 2:
        data += b'\x00'
    total = 0
    for index in range(0, len(data), 2):
        total += (data[index] << 8) | data[index + 1]
        total = (total & 0xFFFF) + (total >> 16)
    return (~total) & 0xFFFF

def parse_http_response(raw):
    head, separator, body = raw.partition(b'\r\n\r\n')
    if not separator:
        raise ValueError('missing HTTP header boundary')
    lines = head.decode('ascii').split('\r\n')
    if lines[0] != 'HTTP/1.0 200 OK':
        raise ValueError('unsupported status line')
    headers = {}
    for line in lines[1:]:
        key, value = line.split(':', 1)
        headers[key.lower()] = value.strip()
    if int(headers.get('content-length', '-1')) != len(body):
        raise ValueError('body length mismatch')
    return body.decode('ascii')

assert translate(0x40AA) == 0x02AA
device = BlockDevice()
filesystem = TinyFS(device)
filesystem.put('boot.bin', machine_image, 1)
assert filesystem.get('boot.bin') == machine_image
packet_body = b'\x00\x00' + machine_image
checksum = internet_checksum(packet_body)
packet = checksum.to_bytes(2, 'big') + machine_image
assert internet_checksum(packet) == 0
response = b'HTTP/1.0 200 OK\r\nContent-Length: 4\r\n\r\nBOOT'
browser_text = parse_http_response(response)
assert browser_text == 'BOOT'
artifact = {
    'schema_version': 1,
    'machine_image_hex': machine_image.hex(),
    'page_table': page_table,
    'translated_example': translate(0x40AA),
    'directory': filesystem.directory,
    'block_size': device.block_size,
    'boot_block_hex': device.read(1).hex(),
    'packet_hex': packet.hex(),
    'checksum': checksum,
    'browser_text': browser_text,
}
target = ARTIFACT_ROOT / 'fromthetransistor/04-system-image.json'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(json.dumps(artifact, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print('system boundaries traced:', list(filesystem.directory), browser_text)


## Exercise and integration failures

1. Predict the physical address for `0x40ff`; then require faults for an unmapped page and a write to the code page.
2. Corrupt one packet bit and show why checksum verification changes.
3. Change `Content-Length` without changing the body and require a parser failure.
4. Add a second file whose payload crosses a block boundary; define allocation and rollback before implementing it.

In [ ]:
assert translate(0x40FF) == 0x02FF
for bad_call in (lambda: translate(0x4100), lambda: translate(0x4000, access='write')):
    try:
        bad_call()
        raise AssertionError('invalid memory access accepted')
    except (MemoryError, PermissionError):
        pass
corrupt = bytearray(packet)
corrupt[-1] ^= 1
assert internet_checksum(bytes(corrupt)) != 0
try:
    parse_http_response(response.replace(b'Content-Length: 4', b'Content-Length: 5'))
    raise AssertionError('bad HTTP length accepted')
except ValueError:
    pass
print('system integration self-checks passed')
